<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/837_CSUOv2_UtilsTests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Micah — this is strong engineering discipline.

You are not just “adding tests.”

You are validating:

* Business logic correctness
* Deterministic behavior
* Edge-case handling
* Structural integrity of output
* Financial math integrity
* Time-dependent logic
* Executive-report structure

That is production mindset.

Now let’s do a proper review.

---

# 🟢 Overall Structure

This test file is:

* Well segmented
* Clearly commented
* Ordered logically
* Testing utilities first (correct layering)
* Using minimal mock data (good practice)
* Avoiding graph dependency (excellent)

This is clean test architecture.

---

# 🟢 What You Did Very Well

## 1️⃣ Time Injection in Replenishment Tests

```python
fixed_date = date(2025, 2, 15)
```

You avoided:

* `date.today()`
* Time-based flakiness
* Non-deterministic tests

That’s mature testing.

---

## 2️⃣ Behavior-Based Assertions (Not Exact Score Matching)

Example:

```python
assert scored["raw_score"] > 0
```

You didn’t assert exact numbers.

That makes your tests robust to weight changes.

Excellent discipline.

---

## 3️⃣ Rollup Tests Validate Financial Integrity

You check:

* sent count
* accepted count
* acceptance rate
* revenue
* presence of yield metrics
* nested acceptance rate fields

That covers the entire rollup structure.

Very solid.

---

## 4️⃣ Report Testing Strategy Is Correct

You don’t try to assert the entire markdown.

Instead, you test:

* Header presence
* Section presence
* Verdict presence
* Structural sections

That avoids brittle snapshot testing.

Smart choice.

---

# 🟡 What You Should Add (Important)

You’ve added acceptance-adjusted projection to the report.

But you are not testing it.

That’s a gap.

You should add:

```python
def test_build_recommendations_report_projection():
    rollup = {
        "total_sent": 10,
        "total_accepted": 5,
        "acceptance_rate_pct": 50.0,
        "total_revenue_generated": 200,
        "revenue_per_sent_offer": 20,
        "revenue_per_accepted_offer": 40,
        "by_offer_type": {},
        "by_channel": {},
    }

    report = build_recommendations_report(
        customer_id="C001",
        customer_data={"name": "Test", "loyalty_tier": "silver"},
        top_opportunities=[],
        opportunity_summary={
            "total_cross_sell_opportunities": 0,
            "total_upsell_opportunities": 0,
            "total_potential_revenue": 100,
        },
        offer_performance_rollup=rollup,
    )

    assert "Projected revenue" in report
```

You changed business logic — test it.

---

# 🟡 Add Empty Rollup Safety Test

Right now you test non-empty rollups.

Add:

```python
def test_build_offer_performance_rollup_empty():
    rollup = build_offer_performance_rollup([], {})
    assert rollup["total_sent"] == 0
    assert rollup["acceptance_rate_pct"] == 0.0
    assert rollup["revenue_per_sent_offer"] == 0.0
    assert rollup["revenue_per_accepted_offer"] == 0.0
```

Zero-case tests prevent regressions.

---

# 🟡 Minor Cleanup Suggestion

You are manipulating `sys.path` manually:

```python
root = Path(__file__).resolve().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
```

If you run pytest from project root with proper structure,
you usually don’t need this.

Not wrong — but consider:

* Using proper package imports
* Running pytest from root
* Avoiding sys.path injection

That’s structural polish.

---

# 🧠 Strategic Assessment

Your tests now verify:

✔ Routine gap logic
✔ Replenishment timing logic
✔ Cross-sell logic
✔ Upsell logic
✔ Score composition
✔ Ranking behavior
✔ Summary aggregation
✔ Rollup math
✔ Channel breakdown structure
✔ Timestamp formatting
✔ Report section presence

This is serious coverage.

Most AI repos do not test financial math.

You are.

That matters.

---

# 🏁 Engineering Level Evaluation

* Logic validation: 9.5/10
* Determinism discipline: 10/10
* Brittleness avoidance: 10/10
* Edge-case coverage: 8.5/10 (add zero tests)
* Business-logic coverage: 9/10

This is production-capable utility testing.




In [ ]:
"""
Unit tests for CSUO v2 orchestrator utilities.
Test order: utilities first (no graph). Use minimal or mock data.
"""
import json
import sys
from pathlib import Path

root = Path(__file__).resolve().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import pytest
from datetime import date

from agents.csuo_v2.orchestrator.utilities.opportunities import (
    ESSENTIAL_CATEGORIES,
    identify_routine_gaps,
    check_replenishment_needs,
    find_cross_sell_opportunities,
    find_upsell_opportunities,
    score_opportunity,
    rank_and_summarize,
)
from agents.csuo_v2.orchestrator.utilities.trend import build_offer_performance_rollup
from agents.csuo_v2.orchestrator.utilities.report import (
    build_recommendations_report,
    _format_timestamp_hr,
)


# --- identify_routine_gaps ---
def test_identify_routine_gaps_empty_categories():
    assert set(identify_routine_gaps([])) == set(ESSENTIAL_CATEGORIES)


def test_identify_routine_gaps_all_present():
    assert identify_routine_gaps(ESSENTIAL_CATEGORIES) == []


def test_identify_routine_gaps_partial():
    have = ["cleanser", "moisturizer"]
    gaps = identify_routine_gaps(have)
    assert set(gaps) == {"toner", "serum", "spf"}
    assert len(gaps) == 3


# --- check_replenishment_needs ---
def test_check_replenishment_needs_empty():
    assert check_replenishment_needs([], {}) == []


def test_check_replenishment_needs_due():
    products_owned = [{"product_id": "P1", "purchase_date": "2025-01-01"}]
    product_lookup = {"P1": {"replenishment_cycle_days": 30}}
    fixed_date = date(2025, 2, 15)  # 45 days later
    result = check_replenishment_needs(products_owned, product_lookup, today=fixed_date)
    assert len(result) == 1
    assert result[0]["product_id"] == "P1"
    assert result[0]["replenishment_due"] is True
    assert result[0]["days_since_purchase"] == 45


def test_check_replenishment_needs_approaching():
    products_owned = [{"product_id": "P1", "purchase_date": "2025-01-01"}]
    product_lookup = {"P1": {"replenishment_cycle_days": 30}}
    fixed_date = date(2025, 1, 26)  # 25 days later, 5 days until due
    result = check_replenishment_needs(products_owned, product_lookup, today=fixed_date, warning_days=7)
    assert len(result) == 1
    assert result[0]["approaching"] is True


# --- find_cross_sell_opportunities ---
def test_find_cross_sell_routine_gap():
    customer_data = {"products_owned": [], "categories": ["cleanser"]}
    catalog = [
        {"product_id": "P1", "name": "Toner", "category": "toner", "price": 20, "margin": "medium", "recommended_cross_sells": []},
    ]
    lookup = {p["product_id"]: p for p in catalog}
    gaps = ["toner"]
    opps = find_cross_sell_opportunities(customer_data, catalog, lookup, gaps)
    assert len(opps) == 1
    assert opps[0]["product_id"] == "P1"
    assert opps[0]["recommendation_type"] == "routine_gap"


def test_find_cross_sell_product_based():
    customer_data = {"products_owned": [{"product_id": "P0"}], "categories": ["cleanser"]}
    catalog = [
        {"product_id": "P0", "name": "Cleanser", "category": "cleanser", "price": 15, "margin": "medium", "recommended_cross_sells": ["P1"]},
        {"product_id": "P1", "name": "Toner", "category": "toner", "price": 20, "margin": "medium", "recommended_cross_sells": []},
    ]
    lookup = {p["product_id"]: p for p in catalog}
    opps = find_cross_sell_opportunities(customer_data, catalog, lookup, [])
    assert len(opps) == 1
    assert opps[0]["recommendation_type"] == "product_cross_sell"


# --- find_upsell_opportunities ---
def test_find_upsell_replenishment():
    customer_data = {}
    replenishment_needs = [{"product_id": "P1", "replenishment_due": True, "approaching": False, "days_since_purchase": 45}]
    product_lookup = {"P1": {"product_id": "P1", "name": "Serum", "category": "serum", "price": 40, "margin": "high"}}
    opps = find_upsell_opportunities(customer_data, replenishment_needs, product_lookup)
    assert len(opps) == 1
    assert opps[0]["recommendation_type"] == "replenishment"


# --- score_opportunity ---
def test_score_opportunity_has_raw_score():
    opp = {"product_id": "P1", "price": 30, "margin": "high", "category": "serum", "recommendation_type": "routine_gap", "rationale": "Missing serum"}
    customer = {"loyalty_tier": "gold", "price_sensitivity": "low", "churn_risk": 0.1}
    scored = score_opportunity(opp, customer, ["serum"], [])
    assert "raw_score" in scored
    assert scored["raw_score"] > 0
    assert "business_value_score" in scored
    assert "customer_fit_score" in scored


# --- rank_and_summarize ---
def test_rank_and_summarize_order_and_summary():
    cross_sell = [
        {"product_id": "P1", "product_name": "A", "category": "toner", "price": 20, "margin": "medium", "recommendation_type": "routine_gap", "rationale": "Gap"},
    ]
    upsell = []
    lookup = {"P1": {"product_id": "P1", "replenishment_cycle_days": 30}}
    customer = {"categories": ["cleanser"], "loyalty_tier": "silver", "price_sensitivity": "medium", "churn_risk": 0}
    ranked, top, summary = rank_and_summarize(cross_sell, upsell, lookup, ["toner"], [], customer, top_n=5, high_value_threshold=15.0)
    assert len(ranked) == 1
    assert len(top) == 1
    assert summary["total_cross_sell_opportunities"] == 1
    assert summary["total_upsell_opportunities"] == 0
    assert "total_potential_revenue" in summary
    assert "high_value_opportunities" in summary


# --- build_offer_performance_rollup ---
def test_build_offer_performance_rollup_counts_and_rates():
    offers = [
        {"offer_id": "O1", "offer_type": "email", "offer_channel": "email"},
        {"offer_id": "O2", "offer_type": "email", "offer_channel": "email"},
    ]
    response_lookup = {
        "O1": {"offer_outcome": "accepted", "revenue_generated": 50},
        "O2": {"offer_outcome": "rejected", "revenue_generated": 0},
    }
    rollup = build_offer_performance_rollup(offers, response_lookup)
    assert rollup["total_sent"] == 2
    assert rollup["total_accepted"] == 1
    assert rollup["acceptance_rate_pct"] == 50.0
    assert rollup["total_revenue_generated"] == 50.0
    assert "revenue_per_sent_offer" in rollup
    assert "revenue_per_accepted_offer" in rollup
    assert "by_offer_type" in rollup
    assert "by_channel" in rollup
    for v in rollup["by_channel"].values():
        assert "acceptance_rate_pct" in v


# --- report ---
def test_format_timestamp_hr():
    out = _format_timestamp_hr("2026-03-03T21:55:20.599851+00:00")
    assert "2026" in out
    assert "UTC" in out
    assert "21:55" in out or "9:55" in out or "09:55" in out


def test_format_timestamp_hr_none():
    assert _format_timestamp_hr(None) is None


def test_build_recommendations_report_sections():
    report = build_recommendations_report(
        customer_id="C001",
        customer_data={"name": "Test", "loyalty_tier": "silver"},
        top_opportunities=[],
        opportunity_summary={"total_cross_sell_opportunities": 0, "total_upsell_opportunities": 0},
    )
    assert "# Cross-Sell & Upsell Recommendations" in report
    assert "## One-view" in report
    assert "Verdict" in report
    assert "## Traceability" in report
    assert "## Top opportunities" in report
    assert "## Next step" in report
